# Ejercicio 5: Espacio Vectorial

Implementación de un Sistema de Recuperación de Información con:

1. Carga del corpus desde Kaggle
2. Preprocesamiento
3. Recuperación con TF-IDF
4. Recuperación con BM25
5. Comparación de resultados

# Parte 0: Carga del Corpus
### Actividad

Carga el corpus   
Realiza las etapas de preprocesamiento sobre el corpus


## 1. Instalación de dependencias

In [1]:
!pip install kaggle pandas nltk scikit-learn rank-bm25

Defaulting to user installation because normal site-packages is not writeable


## 2. Importación de librerías

In [2]:
import os
import re
import zipfile
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 2. Cargar el cvs

In [ ]:
import pandas as pd

# Cargar el archivo CSV y guardarlo en un datagrama
df = pd.read_csv("wikipedia_text_corpus.csv")

# Ver las primeras filas
df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


In [ ]:
# Descargar los recursos necesarios de NLTK

import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LabP5E004\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import re
from nltk.corpus import stopwords
# Eliminamos las palabras mas comunes en ingles.
stop_words = set(stopwords.words('english'))

# Creamos la funcion para tokenizar.
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text) # Eliminamos los caracteres especiales
    tokens = text.split()   # Tokenización simple
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

In [9]:
processed_docs = [preprocess(doc) for doc in documents]
print("Preprocesamiento completado.")

Preprocesamiento completado.


# Parte 1: Recuperación con TF-IDF
### Actividad:

Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF   
A partir de un conjunto de 10 queries, verifica la recuperación del sistema


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1. Crear la matriz TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_docs)

print("Matriz TF-IDF creada.")
print("Dimensiones:", tfidf_matrix.shape)

# 2. Definimos las 10 consultas queries
queries = [
    "artificial intelligence",
#    "machine learning",
#    "computer networks",
#    "operating systems",
    "database systems",
    "cyber security",
    "probability and statistics",
    "software engineering",
    "data structures",
    "java programming",
    "computerized analog",
    "biological treatment",
    "Customer knowledge"

]



Matriz TF-IDF creada.
Dimensiones: (5000, 84633)


In [ ]:
# 3. Función de búsqueda
def search_tfidf(query, top_k=5):
    query_processed = preprocess(query)
    query_vector = vectorizer.transform([query_processed])

    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_k]

    # 
    results = []
    for idx in top_indices:
        results.append({
            "documento": idx,
            "score": similarities[idx],
            "texto": documents[idx][:200]
        })

    return pd.DataFrame(results)



In [20]:
# 4. Probar las 10 consultas
tfidf_results = {}

for query in queries:
    print("=" * 80)
    print("QUERY:", query)
    result = search_tfidf(query)
    tfidf_results[query] = result
    display(result)

QUERY: artificial intelligence


,documento,score,texto
0,3905,0.470889,Accounting intelligence\n\nA specialist form o...
1,4563,0.456732,Intelligence engine\n\nAn intelligence engine ...
2,669,0.403018,Open Letter on Artificial Intelligence\n\nIn J...
3,2618,0.352794,Artificial general intelligence\n\nArtificial ...
4,485,0.342451,Strategic Computing Initiative\n\nThe United S...


QUERY: operating systems


,documento,score,texto
0,3532,0.595812,List of operating systems\n\nThis is a list of...
1,4955,0.270599,Embedded system\n\nAn embedded system is a pro...
2,174,0.260313,Information system\n\nInformation systems (IS)...
3,1687,0.231710,ACM SIGOPS\n\nACM SIGOPS is the Association fo...
4,3578,0.215268,Improvement Support Systems\n\nImprovement Sup...


QUERY: database systems


...

,documento,score,texto
0,4473,0.494693,Real time Java\n\nReal time Java is a catch-al...
1,2169,0.331344,Psychology of programming\n\nThe psychology of...
2,613,0.283718,List of programming languages\n\nThe aim of th...
3,465,0.281477,"IGeoSIT\n\niGeoSIT, the Interim Geo-Spatial In..."
4,2513,0.254399,Skeleton (computer programming)\n\nSkeleton pr...


QUERY: computerized analog


,documento,score,texto
0,1001,0.267137,Digital television transition\n\nThe digital t...
1,3578,0.202765,Improvement Support Systems\n\nImprovement Sup...
2,641,0.186352,Analog forestry\n\nAnalog forestry is an appro...
3,1446,0.180060,"Digital-to-analog converter\n\nIn electronics,..."
4,2544,0.174426,Jim Williams (analog designer)\n\nJames M. Wil...


QUERY: biological treatment


,documento,score,texto
0,3504,0.472917,Waste treatment\n\nWaste treatment refers to t...
1,2664,0.406746,Wastewater treatment\n\nwaste water treatment ...
2,3687,0.315601,Mechanical heat treatment\n\nMechanical heat t...
3,1161,0.265114,List of solid waste treatment technologies\n\n...
4,83,0.245892,Mechanical biological treatment\n\nA mechanica...


# Parte 2: Recuperación con BM25
### Actividad:

Implementa un sistema de recuperación usando el modelo BM25.
Para el mismo conjunto de 10 queries, verifica la recuperación del sistema


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np
import pandas as pd

# 1. Tokenizar los documentos preprocesados
tokenized_docs = [doc.split() for doc in processed_docs]

# 2. Crear el modelo BM25
bm25 = BM25Okapi(tokenized_docs)

print("Modelo BM25 creado correctamente.")

Modelo BM25 creado correctamente.


In [ ]:
# 3. Función de búsqueda con BM25
def search_bm25(query, top_k=5):
    # Preprocesar la consulta y convertirla en tokens
    query_tokens = preprocess(query).split()

    # Calcular puntajes de relevancia
    scores = bm25.get_scores(query_tokens)

    # Obtener los índices de los documentos más relevantes
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Construir resultados
    results = []
    for idx in top_indices:
        results.append({
            "documento": int(idx),
            "score": float(scores[idx]),
            "texto": documents[idx][:200]
        })

    return pd.DataFrame(results)

In [16]:
# 4. Probar las mismas 10 consultas
bm25_results = {}

for query in queries:
    print("=" * 80)
    print("QUERY:", query)
    result = search_bm25(query)
    bm25_results[query] = result
    display(result)

QUERY: artificial intelligence


,documento,score,texto
0,669,14.292287,Open Letter on Artificial Intelligence\n\nIn J...
1,485,13.857551,Strategic Computing Initiative\n\nThe United S...
2,2618,13.608052,Artificial general intelligence\n\nArtificial ...
3,3799,13.148273,Stochastic neural analog reinforcement calcula...
4,3226,12.238894,Hypothetical technology\n\nHypothetical techno...


QUERY: machine learning


,documento,score,texto
0,4733,12.272041,Teaching machine\n\nTeaching machines were ori...
1,3498,11.831675,Sidney L. Pressey\n\nSidney Leavitt Pressey (B...
2,466,11.598824,Information engineering (field)\n\nInformation...
3,1892,10.933898,Data stream mining\n\nData Stream Mining is th...
4,4457,10.876585,GP5 chip\n\nThe GP5 is a co-processor accelera...


QUERY: computer networks


...

,documento,score,texto
0,1904,8.231163,Institute of Software Engineers\n\nThe Institu...
1,193,8.053967,Software diagnosis\n\nSoftware diagnosis (also...
2,4009,7.972742,Computing\n\nComputing is any activity that us...
3,824,7.905550,Software development\n\nSoftware development i...
4,855,7.902934,Hungarian Testing Board\n\nThe Hungarian Testi...


QUERY: data structures


,documento,score,texto
0,3487,8.472344,Ocean data acquisition system\n\nAn ocean data...
1,3635,7.607777,"PEARL (programming language)\n\nPEARL, or Proc..."
2,407,7.599669,Directed assembly of micro- and nano-structure...
3,4816,7.525688,Integrated computational materials engineering...
4,1892,7.503254,Data stream mining\n\nData Stream Mining is th...


QUERY: java programming


,documento,score,texto
0,4473,16.975028,Real time Java\n\nReal time Java is a catch-al...
1,2513,15.331149,Skeleton (computer programming)\n\nSkeleton pr...
2,1023,13.972967,Simon Phipps (programmer)\n\nSimon Phipps is a...
3,722,12.881870,"Programmer\n\nA programmer, developer, dev, co..."
4,1128,12.739243,Standard Performance Evaluation Corporation\n\...


# Parte 3: Comparación de resultados
### Actividad:

Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación


In [17]:
comparison = []

for query in queries:
    tfidf_docs = tfidf_results[query]["documento"].tolist()
    bm25_docs = bm25_results[query]["documento"].tolist()

    comparison.append({
        "query": query,
        "tfidf_top5": tfidf_docs,
        "bm25_top5": bm25_docs,
        "coincidencias": len(set(tfidf_docs) & set(bm25_docs))
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

,query,tfidf_top5,bm25_top5,coincidencias
0,artificial intelligence,"[3905, 4563, 669, 2618, 485]","[669, 485, 2618, 3799, 3226]",3
1,machine learning,"[4733, 51, 466, 3227, 3498]","[4733, 3498, 466, 1892, 4457]",3
2,computer networks,"[15, 4009, 4553, 1341, 2923]","[1616, 1341, 270, 4009, 3328]",2
3,operating systems,"[3532, 4955, 174, 1687, 3578]","[3532, 1687, 740, 4955, 1970]",3
4,database systems,"[1158, 2577, 2963, 2060, 4410]","[2577, 2060, 2947, 2213, 1700]",2
5,cyber security,"[2924, 4258, 699, 3947, 3841]","[3841, 3431, 4258, 2924, 2057]",3
6,probability and statistics,"[3799, 2296, 3709, 3138, 2493]","[2296, 3709, 3180, 1591, 466]",2
7,software engineering,"[4982, 193, 824, 1477, 711]","[1904, 193, 4009, 824, 855]",2
8,data structures,"[1700, 4604, 3616, 521, 1892]","[3487, 3635, 407, 4816, 1892]",1
9,java programming,"[4473, 2169, 613, 465, 2513]","[4473, 2513, 1023, 722, 1128]",2
